# 03 — Significance Testing & Error Analysis

Addresses reviewer points **#3 (statistical significance)**, **#5 (comparison
with prior work)** and **#6 (error breakdown)**, plus the *deeper analysis of
model limitations* request.

Consumes only the `oof_*.jsonl` files written by notebook 02. Nothing is
retrained here, so the notebook runs in seconds and can be re-run freely.

**Why out-of-fold predictions matter.** Every analysis below is computed on
predictions the model made for sentences it did not train on. The error analysis
in the submitted version used the production model — fine-tuned on the whole
gold corpus — and scored it on that same corpus, so the reported errors were
training-set errors. That is fixed here.

In [ ]:
# --- identical bootstrap cell -----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pytorch-crf seqeval safetensors
!cp /content/drive/MyDrive/crypto-ner/code/*.py /content/

import sys; sys.path.append('/content')
from config import *
bootstrap()

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ROOT = /content/drive/MyDrive/crypto-ner
device: CPU only

  Inputs:
    [ok     ] gold_standard_bio_labeled.csv         0.9 MB  nb 01, 02, 03
    [ok     ] silver_standard_bio.csv              17.5 MB  nb 01
    [ok     ] silver_standard_dataset.csv          14.3 MB  nb 01
    [ok     ] lkb_cleaned.json                      0.2 MB  nb 01, 02, 03
    [ok     ] lkb_case_sensitive.json               0.0 MB  nb 01, 02, 03
    [ok     ] dataset_mentah_twitter.csv           16.4 MB  nb 01, funnel only

  01 → 02
    [ok     ] silver_standard_bio_naive.csv      naive silver labels, required by the -HD ablation

  01 → 03
    [ok     ] gold_strata.csv                    per-sentence stratum, used in the error breakdown

  02 → 03
    [ok     ] oof_proposed.jsonl                 out-of-fold predictions, proposed system
    [ok     ] oof_naive_silver

True

In [ ]:
import os, json, itertools
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

import utils
from utils import (load_oof, entity_f1, entity_counts, prf, extract_entities,
                   count_bio_violations, paired_bootstrap, approximate_randomization,
                   bootstrap_ci, holm_bonferroni, load_lkb, BLUE_CHIPS, load_bio_csv, get_folds)

# Display order for the paper table
SYSTEMS = [
    ('gazetteer',             'Gazetteer longest-match (LKB only)'),
    ('bilstm_crf',            'BiLSTM-CRF'),
    ('mbert_crf_gold',        'mBERT-CRF (gold only)'),
    ('indobert_crf_gold',     'IndoBERT-CRF (gold only)'),
    ('indobertweet_crf_gold', 'IndoBERTweet-CRF (gold only)  [-DS]'),
    ('base_only',             'Base model, no fine-tuning     [-FT]'),
    ('naive_silver_ft',       'Proposed w/o denoising         [-HD]'),
    ('softmax_ft',            'Proposed w/o CRF               [-CRF]'),
    ('proposed',              'Proposed (IndoBERTweet-CRF, DS+HD+FT)'),
]
REFERENCE = 'proposed'

data = {}
for key, _ in SYSTEMS:
    p = os.path.join(OOF, f'oof_{key}.jsonl')
    if os.path.exists(p):
        data[key] = load_oof(p)
    else:
        print(f'[missing] {key}')

assert REFERENCE in data, (
    'oof_proposed.jsonl not found — run notebook 02 before this one.')
Y_TRUE = [r['true'] for r in data[REFERENCE]]
FOLDS  = np.array([r['fold'] for r in data[REFERENCE]])
IDS    = [r['sentence_id'] for r in data[REFERENCE]]
TOKENS = [r['tokens'] for r in data[REFERENCE]]
print(f'{len(data)} systems loaded, {len(Y_TRUE)} sentences each')

9 systems loaded, 1461 sentences each


In [ ]:
gold_ids, gold_sents, gold_labels = load_bio_csv(P_GOLD_BIO)
splits, fold_of = get_folds(len(gold_sents))

print(f'Gold sentences : {len(gold_sents)}')
for f, (tr, te) in enumerate(splits):
    print(f'  fold {f+1}: train={len(tr)}  test={len(te)}')

lkb_unamb = load_lkb(P_LKB_UNAMB)
lkb_amb   = load_lkb(P_LKB_AMB)

Gold sentences : 1461
  fold 1: train=1168  test=293
  fold 2: train=1169  test=292
  fold 3: train=1169  test=292
  fold 4: train=1169  test=292
  fold 5: train=1169  test=292


In [ ]:
LKB_ALL = lkb_unamb | lkb_amb
moved = [' '.join(t[s:e]) for t, y in zip(TOKENS, Y_TRUE)
         for (s, e, _) in extract_entities(y)
         if ' '.join(t[s:e]).lower().lstrip('$#') not in lkb_unamb
         and ' '.join(t[s:e]).lower().lstrip('$#') in LKB_ALL]
print(f'{len(moved)} entitas gold pindah dari OOV ke In-LKB')
print(Counter(m.lower() for m in moved).most_common(15))

28 entitas gold pindah dari OOV ke In-LKB
[('bnb', 12), ('$bnb', 5), ('#bnb', 4), ('ab', 2), ('$gas', 2), ('cyber', 2), ('$soon', 1)]


counterfactual blacklist

In [ ]:
amb, unamb = load_lkb(P_LKB_AMB), load_lkb(P_LKB_UNAMB)
shadowed = amb & unamb
_, sents, _ = load_bio_csv(P_SILVER_BIO)

import collections
extra = collections.Counter()
for toks in sents:
    for t in toks:
        core = t.lstrip('$#').lower()
        if core in shadowed and not t.isupper():
            extra[core] += 1
print('kemunculan huruf kecil dari surface ter-shadow:', sum(extra.values()))
print(extra.most_common(25))

kemunculan huruf kecil dari surface ter-shadow: 14790
[('ada', 5259), ('hype', 1981), ('mana', 718), ('gua', 678), ('sol', 591), ('solana', 548), ('in', 459), ('trump', 373), ('on', 342), ('up', 296), ('pump', 262), ('open', 227), ('real', 209), ('momentum', 173), ('link', 157), ('btw', 151), ('super', 119), ('la', 114), ('hyperliquid', 114), ('era', 108), ('pro', 100), ('me', 100), ('not', 89), ('om', 87), ('ath', 83)]


In [ ]:
import collections
ids_d, sents, labs_den = load_bio_csv(P_SILVER_BIO)
ids_n, _,     labs_nai = load_bio_csv(P_SILVER_BIO_NAIVE)
assert ids_d == ids_n

amb_only = load_lkb(P_LKB_AMB) - load_lkb(P_LKB_UNAMB)   # yang benar-benar diproses

kept, dropped = collections.Counter(), collections.Counter()
for toks, ld, ln in zip(sents, labs_den, labs_nai):
    for t, a, b in zip(toks, ld, ln):
        core = t.lstrip('$#').lower()
        if core not in amb_only or b == 'O':
            continue
        (kept if a != 'O' else dropped)[core] += 1

print('dipertahankan:', sum(kept.values()), '| ditekan:', sum(dropped.values()))
for s in sorted(set(kept) | set(dropped), key=lambda s: -(kept[s] + dropped[s]))[:20]:
    tot = kept[s] + dropped[s]
    print(f'  {s:12s} pertahan {kept[s]:6d}  tekan {dropped[s]:6d}  ({100*kept[s]/tot:5.1f}% dipertahankan)')

dipertahankan: 438 | ditekan: 15404
  ini          pertahan     39  tekan   9115  (  0.4% dipertahankan)
  amp          pertahan      0  tekan   2033  (  0.0% dipertahankan)
  dia          pertahan      8  tekan   1375  (  0.6% dipertahankan)
  bnb          pertahan    154  tekan    710  ( 17.8% dipertahankan)
  meme         pertahan     18  tekan    443  (  3.9% dipertahankan)
  gt           pertahan      6  tekan    391  (  1.5% dipertahankan)
  memecoin     pertahan     40  tekan    313  ( 11.3% dipertahankan)
  gas          pertahan     11  tekan    239  (  4.4% dipertahankan)
  cash         pertahan     11  tekan    110  (  9.1% dipertahankan)
  max          pertahan      0  tekan    112  (  0.0% dipertahankan)
  move         pertahan      9  tekan     57  ( 13.6% dipertahankan)
  core         pertahan      7  tekan     53  ( 11.7% dipertahankan)
  flow         pertahan      0  tekan     49  (  0.0% dipertahankan)
  ab           pertahan     35  tekan     10  ( 77.8% dipertahankan

In [ ]:
amb = load_lkb(P_LKB_AMB)
n = sum(1 for toks, y in zip(gold_sents, gold_labels)
        for (s, e, _) in extract_entities(y)
        if ' '.join(toks[s:e]).lstrip('$#').lower() in amb)
print(f'{n} dari 1182 entitas gold ({100*n/1182:.1f}%) bersurface tier Ambiguous')

130 dari 1182 entitas gold (11.0%) bersurface tier Ambiguous


## A. Main results table

One table covering baselines, ablations, and significance. `ΔF1` and `p` are
measured against the proposed system.

Two F1 columns are reported because they answer different questions. The
per-fold mean ± std shows stability; the pooled micro-F1 over all 1,461
out-of-fold predictions is the single number the significance test operates on.
They will not be identical — F1 is not a linear average — so quote both and
never mix them.

In [ ]:
import time
N_RESAMPLES = 100000     # ~10s for the whole table; raise freely if you want
_t0 = time.time()

def per_fold_scores(y_pred):
    out = []
    for f in sorted(set(FOLDS)):
        idx = np.where(FOLDS == f)[0]
        out.append(entity_f1([Y_TRUE[i] for i in idx], [y_pred[i] for i in idx]))
    return out

ref_pred = [r['pred'] for r in data[REFERENCE]]
rows, pvals, pkeys = [], [], []

for key, label in SYSTEMS:
    if key not in data:
        continue
    pred = [r['pred'] for r in data[key]]
    pooled = entity_f1(Y_TRUE, pred)
    pf = per_fold_scores(pred)
    f1s = [m['f1'] for m in pf]
    lo, hi = bootstrap_ci(Y_TRUE, pred, n_resamples=N_RESAMPLES)

    row = {'System': label,
           'P': round(pooled['precision'], 4),
           'R': round(pooled['recall'], 4),
           'F1': round(pooled['f1'], 4),
           '95% CI': f'[{lo:.3f}, {hi:.3f}]',
           'F1 per-fold': f'{np.mean(f1s):.4f} ± {np.std(f1s):.4f}'}

    if key != REFERENCE:
        bs = paired_bootstrap(Y_TRUE, pred, ref_pred, n_resamples=N_RESAMPLES)
        row['dF1 vs proposed'] = round(-bs['delta_f1'], 4)
        pvals.append(bs['p_value']); pkeys.append(key)
        row['_p_raw'] = bs['p_value']
    else:
        row['dF1 vs proposed'] = '—'
        row['_p_raw'] = np.nan
    rows.append(row)

adj, reject = holm_bonferroni(np.array(pvals))
adj_map = dict(zip(pkeys, adj))
order = [k for k, _ in SYSTEMS if k in data]
for r, k in zip(rows, order):
    r['p (Holm)'] = '—' if k == REFERENCE else (
        '<0.001' if adj_map[k] < 0.001 else f'{adj_map[k]:.3f}')
    r.pop('_p_raw', None)

df_main = pd.DataFrame(rows)
df_main.to_csv(f'{OUT}/table_main_results.csv', index=False)
print(f'{len(rows)} systems, {N_RESAMPLES} resamples, {time.time()-_t0:.1f}s')
display(df_main)

9 systems, 100000 resamples, 84.7s


,System,P,R,F1,95% CI,F1 per-fold,dF1 vs proposed,p (Holm)
0,Gazetteer longest-match (LKB only),0.4691,0.8029,0.5922,"[0.564, 0.619]",0.5923 ± 0.0461,-0.2619,<0.001
1,BiLSTM-CRF,0.7887,0.6252,0.6975,"[0.667, 0.726]",0.6980 ± 0.0323,-0.1566,<0.001
2,mBERT-CRF (gold only),0.8227,0.8325,0.8276,"[0.805, 0.849]",0.8272 ± 0.0269,-0.0265,0.028
3,IndoBERT-CRF (gold only),0.8561,0.8553,0.8557,"[0.835, 0.875]",0.8554 ± 0.0106,0.0016,0.852
4,IndoBERTweet-CRF (gold only) [-DS],0.8454,0.7631,0.8021,"[0.777, 0.826]",0.8010 ± 0.0267,-0.0519,<0.001
5,"Base model, no fine-tuning [-FT]",0.4672,0.8020,0.5905,"[0.562, 0.617]",0.5908 ± 0.0405,-0.2636,<0.001
6,Proposed w/o denoising [-HD],0.8191,0.8655,0.8416,"[0.821, 0.861]",0.8412 ± 0.0359,-0.0125,0.052
7,Proposed w/o CRF [-CRF],0.8209,0.7910,0.8057,"[0.782, 0.828]",0.8050 ± 0.0199,-0.0484,<0.001
8,"Proposed (IndoBERTweet-CRF, DS+HD+FT)",0.8328,0.8765,0.8541,"[0.834, 0.873]",0.8533 ± 0.0311,—,—


**Reading the p column.** A non-significant ablation delta is a real finding,
not a failure. If removing the CRF changes nothing measurable, say so — that is
more credible than asserting a benefit the data does not support, and reviewers
reward it. State the test used, the number of resamples, and the correction
applied in the caption.

In [ ]:
# Robustness check: a second test family on the two comparisons that matter most
for key in ['naive_silver_ft', 'softmax_ft']:
    if key not in data:
        continue
    ar = approximate_randomization(Y_TRUE, [r['pred'] for r in data[key]],
                                   ref_pred, n_trials=N_RESAMPLES)
    print(f'{key:22s} approximate randomization p = {ar["p_value"]:.4f}')

naive_silver_ft        approximate randomization p = 0.0243
softmax_ft             approximate randomization p = 0.0000


## B. Structural validity of predicted sequences

The submitted paper claims zero BIO transition violations. Measured on the
production model's own training data, that claim carries little weight. Here it
is measured on out-of-fold predictions, and — crucially — against the softmax
variant, which is the only way the CRF's contribution becomes visible.

In [ ]:
rows = []
for key, label in SYSTEMS:
    if key not in data:
        continue
    bad, total = count_bio_violations([r['pred'] for r in data[key]])
    rows.append({'System': label, 'Invalid transitions': bad,
                 'Tokens': total, 'Rate (%)': round(100 * bad / max(1, total), 4)})
df_bio = pd.DataFrame(rows)
df_bio.to_csv(f'{OUT}/table_bio_violations.csv', index=False)
display(df_bio)

,System,Invalid transitions,Tokens,Rate (%)
0,Gazetteer longest-match (LKB only),0,50242,0.0
1,BiLSTM-CRF,0,50242,0.0
2,mBERT-CRF (gold only),0,50242,0.0
3,IndoBERT-CRF (gold only),0,50242,0.0
4,IndoBERTweet-CRF (gold only) [-DS],0,50242,0.0
5,"Base model, no fine-tuning [-FT]",0,50242,0.0
6,Proposed w/o denoising [-HD],0,50242,0.0
7,Proposed w/o CRF [-CRF],0,50242,0.0
8,"Proposed (IndoBERTweet-CRF, DS+HD+FT)",0,50242,0.0


## C. Error breakdown

Reviewer #6 asked for a breakdown by entity type. The schema has a single type
(`CRYPTO`), so a literal per-type table would have one row. The substance of the
request — *where does the model fail, and on what kind of entity* — is answered
by slicing along four dimensions that do vary.

Say exactly this in the response letter. Reviewers accept the substitution when
the reason is stated plainly and the alternative is more informative.

In [ ]:
strata_path = P_GOLD_STRATA   # handoff from notebook 01
if os.path.exists(strata_path):
    smap = pd.read_csv(strata_path).set_index('sentence_id')['stratum'].to_dict()
    strata = [smap.get(s, 'Other') for s in IDS]
else:
    print('gold_strata.csv not found — recomputing (run notebook 01 to cache it)')
    from utils import assign_strata
    strata = assign_strata(TOKENS, load_lkb(P_LKB_UNAMB), load_lkb(P_LKB_AMB))

rows = []
for s in ['Blue-Chip', 'Altcoin', 'Background', 'Other']:
    idx = [i for i, t in enumerate(strata) if t == s]
    if not idx:
        continue
    m = entity_f1([Y_TRUE[i] for i in idx], [ref_pred[i] for i in idx])
    tp, np_, nt = entity_counts([Y_TRUE[i] for i in idx], [ref_pred[i] for i in idx])
    rows.append({'Stratum': s, 'Sentences': len(idx), 'Gold entities': nt,
                 'P': round(m['precision'], 4), 'R': round(m['recall'], 4),
                 'F1': round(m['f1'], 4)})
df_str = pd.DataFrame(rows)
df_str.to_csv(f'{OUT}/table_error_by_stratum.csv', index=False)
display(df_str)

,Stratum,Sentences,Gold entities,P,R,F1
0,Blue-Chip,387,623,0.8728,0.9470,0.9084
1,Altcoin,754,438,0.7597,0.8082,0.7832
2,Background,313,113,0.8936,0.7434,0.8116
3,Other,7,8,1.0000,1.0000,1.0000


In [ ]:
# Surface-form categories: full name vs ticker vs cashtag/hashtag vs alias
lkb_unamb = load_lkb(P_LKB_UNAMB)
ALIASES = {'betece', 'eter'}

def surface_category(text):
    t = text.lower().strip()
    if text.startswith('$') or text.startswith('#'):
        return 'Cashtag / hashtag'
    if t in ALIASES:
        return 'Community alias'
    if len(t.split()) > 1:
        return 'Multi-token name'
    if len(t) <= 4 and text.isupper():
        return 'Ticker (uppercase)'
    if len(t) <= 4:
        return 'Ticker (lowercase)'
    return 'Full name'

gold_by_cat = Counter(); found_by_cat = Counter()
for toks, gold, pred in zip(TOKENS, Y_TRUE, ref_pred):
    gset = set(extract_entities(gold)); pset = set(extract_entities(pred))
    for span in gset:
        cat = surface_category(' '.join(toks[span[0]:span[1]]))
        gold_by_cat[cat] += 1
        if span in pset:
            found_by_cat[cat] += 1

rows = [{'Surface category': c, 'Gold entities': gold_by_cat[c],
         'Correctly detected': found_by_cat[c],
         'Recall': round(found_by_cat[c] / gold_by_cat[c], 4)}
        for c in sorted(gold_by_cat, key=lambda x: -gold_by_cat[x])]
df_cat = pd.DataFrame(rows)
df_cat.to_csv(f'{OUT}/table_error_by_surface.csv', index=False)
display(df_cat)

,Surface category,Gold entities,Correctly detected,Recall
0,Ticker (uppercase),402,351,0.8731
1,Cashtag / hashtag,334,311,0.9311
2,Full name,303,257,0.8482
3,Ticker (lowercase),128,110,0.8594
4,Multi-token name,15,7,0.4667


In [ ]:
# Span length, and whether the entity exists in the LKB at all (OOV wrt LKB)
LKB_ALL = lkb_unamb | lkb_amb
ref_pred = [r['pred'] for r in data[REFERENCE]]
def bucket_rows(keyfn, title):
    g, f = Counter(), Counter()
    for toks, gold, pred in zip(TOKENS, Y_TRUE, ref_pred):
        gset = set(extract_entities(gold)); pset = set(extract_entities(pred))
        for span in gset:
            k = keyfn(toks, span)
            g[k] += 1
            if span in pset:
                f[k] += 1
    return pd.DataFrame([{title: k, 'Gold': g[k], 'Detected': f[k],
                          'Recall': round(f[k] / g[k], 4)} for k in sorted(g, key=str)])

df_len = bucket_rows(lambda t, s: f'{s[1]-s[0]} token', 'Span length')
df_oov = bucket_rows(
    lambda t, s: 'In LKB' if ' '.join(t[s[0]:s[1]]).lower().lstrip('$#') in LKB_ALL
                 else 'Not in LKB', 'LKB membership')
display(df_len); display(df_oov)
df_len.to_csv(f'{OUT}/table_error_by_span_length.csv', index=False)
df_oov.to_csv(f'{OUT}/table_error_by_lkb.csv', index=False)

,Span length,Gold,Detected,Recall
0,1 token,1167,1029,0.8817
1,2 token,15,7,0.4667


,LKB membership,Gold,Detected,Recall
0,In LKB,956,913,0.9550
1,Not in LKB,226,123,0.5442


## D. Error taxonomy

Counts and percentages for each failure mode, replacing the two anecdotal
examples in the submitted Section IV.D. Keep one or two examples in the paper —
they are readable and memorable — but anchor them to this distribution so the
reader knows how representative they are.

In [ ]:
FIN_MARKERS = set(utils.FINANCIAL_KEYWORDS)

def classify_errors(toks, gold, pred):
    gset = set(extract_entities(gold)); pset = set(extract_entities(pred))
    out = []
    gstarts = {s[0]: s for s in gset}; pstarts = {s[0]: s for s in pset}

    for span in pset - gset:
        surface = ' '.join(toks[span[0]:span[1]])
        if span[0] in gstarts or any(g[0] < span[1] and span[0] < g[1] for g in gset):
            out.append(('Boundary error (FP side)', surface))
        else:
            out.append(('False positive: cross-domain ambiguity', surface))

    ctx = {t.lower() for t in toks}
    for span in gset - pset:
        surface = ' '.join(toks[span[0]:span[1]])
        if span[0] in pstarts or any(p[0] < span[1] and span[0] < p[1] for p in pset):
            out.append(('Boundary error (FN side)', surface))
        elif not (ctx & FIN_MARKERS):
            out.append(('False negative: no slang/jargon context', surface))
        else:
            out.append(('False negative: other', surface))
    return out

tally, examples = Counter(), defaultdict(list)
for toks, gold, pred in zip(TOKENS, Y_TRUE, ref_pred):
    for cat, surf in classify_errors(toks, gold, pred):
        tally[cat] += 1
        if len(examples[cat]) < 8:
            examples[cat].append((surf, ' '.join(toks)[:110]))

total_err = sum(tally.values())
df_tax = pd.DataFrame([{'Error category': c, 'Count': n,
                        '% of errors': round(100 * n / max(1, total_err), 2)}
                       for c, n in tally.most_common()])
df_tax.to_csv(f'{OUT}/table_error_taxonomy.csv', index=False)
display(df_tax)

for cat, ex in examples.items():
    print(f'\n--- {cat} ---')
    for surf, sent in ex[:4]:
        print(f'  [{surf}]  {sent}...')

,Error category,Count,% of errors
0,False positive: cross-domain ambiguity,203,57.34
1,False negative: other,117,33.05
2,False negative: no slang/jargon context,25,7.06
3,Boundary error (FP side),5,1.41
4,Boundary error (FN side),4,1.13



--- False negative: no slang/jargon context ---
  [bnb]  @USER Perang melawan hyperliquid ditambah kejadian 10 Oktober bikin dia jadi public enemy . Jual bnb dan hyper...
  [maskusdc]  @USER Maskbnb bukan maskusdc atau usdt . Udah beli mask aja bli . Bentar lagi bikin huruf W tuh...
  [Maskbnb]  @USER Maskbnb bukan maskusdc atau usdt . Udah beli mask aja bli . Bentar lagi bikin huruf W tuh...
  [btcb]  @USER @USER @USER @USER boleh bridge kalo kamu pny platform bridge yg terpercaya dan murah tapi kalo ga ada ya...

--- False negative: other ---
  [TKO]  @USER Harga TKO hanya naik sebentar Token apa ini ?...
  [pi]  @USER @USER Karena bnyak penipu jadah yg berhsil mencuri menipu koin pi pioneer awam akhirnya lngsung jual aja...
  [MNT]  MNT Reward Booster balik lagi kali ini makin mantep karena ada total 1 . 5 juta $MNT siap dibagikan ! Cuma per...
  [btcnya]  Sudah aku ingatkan jauh jauh hari btc swing aja jgn di hold entitas yg hold btc banyak mereka jual market dump...

--- False po

In [ ]:
# Token-level confusion matrix — cheap, and immediately readable
labels = ['O', 'B-CRYPTO', 'I-CRYPTO']
cm = np.zeros((3, 3), dtype=int)
for gold, pred in zip(Y_TRUE, ref_pred):
    for g, p in zip(gold, pred):
        cm[labels.index(g), labels.index(p)] += 1
df_cm = pd.DataFrame(cm, index=[f'true {l}' for l in labels],
                     columns=[f'pred {l}' for l in labels])
df_cm.to_csv(f'{OUT}/table_confusion.csv')
display(df_cm)

,pred O,pred B-CRYPTO,pred I-CRYPTO
true O,48834,203,8
true B-CRYPTO,142,1040,0
true I-CRYPTO,7,1,7


In [ ]:
# Full error listing for the repository (referenced from the paper, not printed)
rows = []
for sid, toks, gold, pred in zip(IDS, TOKENS, Y_TRUE, ref_pred):
    errs = classify_errors(toks, gold, pred)
    if not errs:
        continue
    rows.append({'Sentence_ID': sid, 'Sentence': ' '.join(toks),
                 'Errors': ' || '.join(f'{c}: {s}' for c, s in errs)})
pd.DataFrame(rows).to_csv(f'{OUT}/error_report_oof.csv', index=False)
print(f'{len(rows)} sentences with at least one error -> error_report_oof.csv')

251 sentences with at least one error -> error_report_oof.csv


## E. Comparison with prior work

Assemble this as **prose in Section IV**, not a table — it costs less space and
avoids presenting incomparable numbers as if they were a head-to-head contest.
The cell below just collects the figures so they are consistent with whatever
the reruns produce.

Points worth making explicitly, since each converts a weakness into analysis:

1. Khairunnisa et al. [12] (94.90%) and FinBERT [16] (91.3%) evaluate on curated
   formal corpora; this work targets informal social media text.
2. Several of those studies report token-level scores; this work uses
   entity-level exact match, a strictly harder criterion.
3. The gold corpus here is 1,461 tweets.
4. Crypto tickers are inherently ambiguous across domains (`eth`, `sol`, `ada`) —
   a problem person/location NER does not face.
5. The 300-tweet Background stratum deliberately depresses precision-friendly
   bias by including financial text with no entity at all.

In [ ]:
prior = pd.DataFrame([
    {'Study': 'Wilie et al. [7]',        'Domain': 'General (NERP)',        'Model': 'IndoBERT',            'F1': 0.7925},
    {'Study': 'Khairunnisa et al. [12]', 'Domain': 'Formal news',           'Model': 'IndoBERT-BiLSTM-CRF', 'F1': 0.9490},
    {'Study': 'Koto et al. [13]',        'Domain': 'Informal Twitter',      'Model': 'IndoBERTweet',        'F1': 0.8540},
    {'Study': 'Wang [16]',               'Domain': 'Financial documents',   'Model': 'FinBERT',             'F1': 0.9130},
])
this_work = entity_f1(Y_TRUE, ref_pred)['f1']
print(prior.to_string(index=False))
print(f'\nThis work (crypto, informal ID Twitter, entity-level exact match): {this_work:.4f}')
print('\nIndicative only — different corpora, domains and evaluation criteria.')

                  Study              Domain               Model     F1
       Wilie et al. [7]      General (NERP)            IndoBERT 0.7925
Khairunnisa et al. [12]         Formal news IndoBERT-BiLSTM-CRF 0.9490
       Koto et al. [13]    Informal Twitter        IndoBERTweet 0.8540
              Wang [16] Financial documents             FinBERT 0.9130

This work (crypto, informal ID Twitter, entity-level exact match): 0.8541

Indicative only — different corpora, domains and evaluation criteria.


## F. Numbers to transfer into the manuscript

| Paper location | Source file |
|---|---|
| Sec. IV.A funnel + corpus stats | `table_funnel.csv`, `table_strata.csv`, `table_silver_vs_gold.csv` (nb 01) |
| Sec. III.E annotator reliability | `table_iaa.csv` (nb 01) |
| Sec. III.F denoising suppression rate | printed by nb 01 section D |
| **Main results table** | `table_main_results.csv` |
| CRF structural validity claim | `table_bio_violations.csv` |
| Sec. IV.D error breakdown | `table_error_taxonomy.csv`, `table_error_by_stratum.csv`, `table_error_by_surface.csv` |
| Repository appendix | `error_report_oof.csv`, `table_confusion.csv`, per-fold results |

Everything not in that first column belongs in the repository, referenced by a
single sentence in the manuscript. That is how the added experiments fit without
pushing the paper over its page allowance.

In [ ]:
import numpy as np, pandas as pd
from utils import load_oof, entity_f1

rows = []
for base in ['proposed', 'naive_silver_ft']:
    f1s, ps, rs = [], [], []
    for s in [42, 1337, 2024]:
        key = base if s == 42 else f'{base}_s{s}'
        r = load_oof(f'{OOF}/oof_{key}.jsonl')
        m = entity_f1([x['true'] for x in r], [x['pred'] for x in r])
        f1s.append(m['f1']); ps.append(m['precision']); rs.append(m['recall'])
    rows.append({'System': base, 'n_seeds': len(f1s),
                 'P mean': round(np.mean(ps), 4), 'R mean': round(np.mean(rs), 4),
                 'F1 mean': round(np.mean(f1s), 4),
                 'F1 std': round(np.std(f1s, ddof=1), 4),
                 'F1 min': round(min(f1s), 4), 'F1 max': round(max(f1s), 4),
                 'F1 per seed': [round(x, 4) for x in f1s]})

df_seed = pd.DataFrame(rows)
df_seed.to_csv(f'{OUT}/table_seed_variance.csv', index=False)
display(df_seed)

deltas = [a - b for a, b in zip(rows[0]['F1 per seed'], rows[1]['F1 per seed'])]
print(f'delta per seed : {[round(d, 4) for d in deltas]}')
print(f'delta mean     : {np.mean(deltas):.4f}  (std {np.std(deltas, ddof=1):.4f})')
print(f'positif di      : {sum(d > 0 for d in deltas)}/3 seed')
print(f'proposed min {rows[0]["F1 min"]:.4f} vs -HD max {rows[1]["F1 max"]:.4f} '
      f'-> rentang {"TIDAK" if rows[0]["F1 min"] > rows[1]["F1 max"] else ""} bertumpang tindih')

,System,n_seeds,P mean,R mean,F1 mean,F1 std,F1 min,F1 max,F1 per seed
0,proposed,3,0.8344,0.8804,0.8568,0.0035,0.8541,0.8607,"[0.8541, 0.8607, 0.8555]"
1,naive_silver_ft,3,0.8222,0.8700,0.8454,0.0033,0.8416,0.8476,"[0.8416, 0.8476, 0.8471]"


delta per seed : [0.0125, 0.0131, 0.0084]
delta mean     : 0.0113  (std 0.0026)
positif di      : 3/3 seed
proposed min 0.8541 vs -HD max 0.8476 -> rentang TIDAK bertumpang tindih


In [ ]:
# Ablation dengan kontrol berpasangan — tiap delta terhadap kontrolnya sendiri
ABLATIONS = [
    ('-HD  (heuristic denoising)',   'naive_silver_ft',       'proposed'),
    ('-DS  (distant supervision)',   'indobertweet_crf_gold', 'proposed'),
    ('-FT  (gold fine-tuning)',      'base_only',             'proposed'),
    ('-CRF (CRF head)',              'softmax_ft',            'indobertweet_crf_gold'),
]

rows, pv = [], []
for label, variant, control in ABLATIONS:
    if variant not in data or control not in data:
        print(f'[lewati] {label}: OOF tidak lengkap');  continue
    v = [r['pred'] for r in data[variant]]
    c = [r['pred'] for r in data[control]]
    bs = paired_bootstrap(Y_TRUE, v, c, n_resamples=N_RESAMPLES)   # delta = F1(c) - F1(v)
    rows.append({'Ablation': label,
                 'Control': control,
                 'F1 full': round(entity_f1(Y_TRUE, c)['f1'], 4),
                 'F1 ablated': round(entity_f1(Y_TRUE, v)['f1'], 4),
                 'Contribution': round(bs['delta_f1'], 4)})
    pv.append(bs['p_value'])

adj, _ = holm_bonferroni(np.array(pv))
for r, a in zip(rows, adj):
    r['p (Holm)'] = '<0.001' if a < 0.001 else f'{a:.3f}'

df_abl = pd.DataFrame(rows)
df_abl.to_csv(f'{OUT}/table_ablation.csv', index=False)
display(df_abl)

# robustness keluarga uji kedua untuk klaim utama
ar = approximate_randomization(Y_TRUE,
        [r['pred'] for r in data['naive_silver_ft']],
        [r['pred'] for r in data['proposed']], n_trials=N_RESAMPLES)
print(f"-HD approximate randomization p = {ar['p_value']:.4f}")

,Ablation,Control,F1 full,F1 ablated,Contribution,p (Holm)
0,-HD (heuristic denoising),proposed,0.8541,0.8416,0.0125,0.052
1,-DS (distant supervision),proposed,0.8541,0.8021,0.0519,<0.001
2,-FT (gold fine-tuning),proposed,0.8541,0.5905,0.2636,<0.001
3,-CRF (CRF head),indobertweet_crf_gold,0.8021,0.8057,-0.0036,0.553


-HD approximate randomization p = 0.0243


In [ ]:
from transformers import AutoTokenizer
n_word = sum(len(t) for t in TOKENS)
for enc in [ENCODER_TWEET, ENCODER_INDOB]:
    tk = AutoTokenizer.from_pretrained(enc)
    n_sub = sum(len(tk.tokenize(' '.join(t))) for t in TOKENS)
    print(f'{enc:42s} {n_sub/n_word:.3f} subword/kata   vocab={tk.vocab_size}')

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

indolem/indobertweet-base-uncased          1.196 subword/kata   vocab=31923


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

indobenchmark/indobert-base-p1             1.239 subword/kata   vocab=30521


In [ ]:
SLANG = {'serok','cuan','nyangkut','rungkad','haka','hodl','fomo','pom','sikat',
         'gan','bre','ngab','gaes','bgt','wkwk','sultan','moon','pump','ath','dump'}

def is_newslike(toks):
    low = {t.lower() for t in toks}
    if low & SLANG:      return False        # ada penanda slang
    if '@USER' in toks:  return False        # register percakapan / reply
    return 'HTTPURL' in toks                 # tautan berita, tanpa penanda percakapan

flags = np.array([is_newslike(t) for t in TOKENS])
SUBSETS = [('News-like', flags), ('Conversational', ~flags)]
MODELS  = [('IndoBERT', 'indobert_crf_gold'),
           ('IndoBERTweet', 'indobertweet_crf_gold'),
           ('Proposed', 'proposed')]

rows = []
for name, mask in SUBSETS:
    idx = np.where(mask)[0]
    yt  = [Y_TRUE[i] for i in idx]
    row = {'Subset': name, 'Sentences': len(idx),
           'Entities': sum(len(extract_entities(y)) for y in yt)}
    for label, key in MODELS:
        p = [r['pred'] for r in data[key]]
        row[label] = round(entity_f1(yt, [p[i] for i in idx])['f1'], 4)
    row['Gap (IB−IBT)'] = round(row['IndoBERT'] - row['IndoBERTweet'], 4)
    rows.append(row)

df_reg = pd.DataFrame(rows)
df_reg.to_csv(f'{OUT}/table_register.csv', index=False)
display(df_reg)

,Subset,Sentences,Entities,IndoBERT,IndoBERTweet,Proposed,Gap (IB−IBT)
0,News-like,338,451,0.8867,0.8409,0.8859,0.0458
1,Conversational,1123,731,0.8360,0.7772,0.8352,0.0588


In [ ]:
from transformers import AutoTokenizer
tk_tweet = AutoTokenizer.from_pretrained(ENCODER_TWEET)
tk_indob = AutoTokenizer.from_pretrained(ENCODER_INDOB)

def fert(tk, items):
    return sum(len(tk.tokenize(x)) for x in items) / max(1, len(items))

surfaces = sorted({' '.join(TOKENS[i][s:e])
                   for i, y in enumerate(Y_TRUE) for (s, e, _) in extract_entities(y)})
allwords = [t for toks in TOKENS for t in toks]

for name, items in [('Surface form entitas', surfaces),
                    ('Leksikon slang kripto', sorted(SLANG)),
                    ('Seluruh token korpus',  allwords)]:
    print(f'{name:24s} IndoBERTweet {fert(tk_tweet, items):.2f}  '
          f'IndoBERT {fert(tk_indob, items):.2f}  subword/item  (n={len(items)})')

Surface form entitas     IndoBERTweet 2.45  IndoBERT 2.40  subword/item  (n=346)
Leksikon slang kripto    IndoBERTweet 1.35  IndoBERT 1.55  subword/item  (n=20)
Seluruh token korpus     IndoBERTweet 1.20  IndoBERT 1.24  subword/item  (n=50242)
